In [1]:
from __future__ import annotations

import csv
import json
import random
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


# ============================================================
# Configuration
# ============================================================

DATASET_ROOT = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical"
    r"\Code\technical validation\ROI_Classification_Dataset"
)

OUTPUT_ROOT = Path(
    r"D:\Submitted Matrial (conference&journal)\Axial Data Artical"
    r"\Code\technical validation\ConvNeXt_ROI_Classification"
)

# Train three separate models.
ANATOMICAL_REGIONS = [
    "Central",
    "Lateral",
    "Foramina",
]

CLASS_NAMES = [
    "Normal",
    "Stenosis",
    "Severe Stenosis",
]

CLASS_FOLDER_TO_ID = {
    "0_normal": 0,
    "1_stenosis": 1,
    "2_severe_stenosis": 2,
}

NUM_CLASSES = 3

# ROI input size: width × height = 64 × 64.
IMAGE_SIZE = (64, 64)

MODEL_NAME = "convnext_tiny"

# ImageNet-pretrained ConvNeXt.
PRETRAINED = True

EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 12

USE_CLASS_WEIGHTS = True
USE_AMP = True

# Select the best checkpoint using validation macro-F1.
BEST_MODEL_METRIC = "macro_f1"

SEED = 42

# Set to True to train the models.
TRAIN_MODELS = True

# Evaluate train, validation, and test after training.
EVALUATE_AFTER_TRAINING = True


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ============================================================
# Dataset
# ============================================================

class ROIDataset(Dataset):
    """
    Dataset for loading ROI PNG images.

    Expected folder structure:

    region/
        train/
            0_normal/
            1_stenosis/
            2_severe_stenosis/
        val/
            ...
        test/
            ...
    """

    SUPPORTED_EXTENSIONS = {
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp",
        ".tif",
        ".tiff",
    }

    def __init__(
        self,
        split_directory: Path,
        transform=None,
    ) -> None:
        self.split_directory = Path(split_directory)
        self.transform = transform

        self.samples: list[tuple[Path, int]] = []

        if not self.split_directory.is_dir():
            raise FileNotFoundError(
                f"Dataset split directory was not found:\n"
                f"{self.split_directory}"
            )

        for class_folder, class_id in CLASS_FOLDER_TO_ID.items():
            class_directory = self.split_directory / class_folder

            if not class_directory.is_dir():
                raise FileNotFoundError(
                    f"Class directory was not found:\n"
                    f"{class_directory}"
                )

            image_paths = sorted(
                path
                for path in class_directory.rglob("*")
                if (
                    path.is_file()
                    and path.suffix.lower() in self.SUPPORTED_EXTENSIONS
                )
            )

            for image_path in image_paths:
                self.samples.append((image_path, class_id))

        if not self.samples:
            raise RuntimeError(
                f"No images were found in:\n{self.split_directory}"
            )

        self.targets = [
            class_id
            for _, class_id in self.samples
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, class_id = self.samples[index]

        # OpenCV loads the original ROI without adding padding or context.
        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if image is None or image.size == 0:
            raise RuntimeError(
                f"Could not read ROI image:\n{image_path}"
            )

        # Convert grayscale MRI ROI to three channels because the
        # pretrained ConvNeXt model expects three-channel input.
        image = cv2.cvtColor(
            image,
            cv2.COLOR_GRAY2RGB,
        )

        image = Image.fromarray(image)

        if self.transform is not None:
            image = self.transform(image)

        return image, class_id, str(image_path)


# ============================================================
# Transformations
# ============================================================

def get_transforms():
    """
    Resize every ROI directly to 64×64.

    No padding, margin, or crop is added.
    """

    train_transform = transforms.Compose(
        [
            transforms.Resize(
                IMAGE_SIZE,
                interpolation=transforms.InterpolationMode.BICUBIC,
                antialias=True,
            ),

            # Mild augmentation suitable for small grayscale MRI ROIs.
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(
                degrees=5,
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=0,
            ),
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10,
            ),

            transforms.ToTensor(),

            # ImageNet normalization for pretrained ConvNeXt.
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )

    evaluation_transform = transforms.Compose(
        [
            transforms.Resize(
                IMAGE_SIZE,
                interpolation=transforms.InterpolationMode.BICUBIC,
                antialias=True,
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )

    return train_transform, evaluation_transform


# ============================================================
# Data loaders
# ============================================================

def create_dataloaders(
    region: str,
    device: torch.device,
):
    train_transform, evaluation_transform = get_transforms()

    region_directory = DATASET_ROOT / region

    train_dataset = ROIDataset(
        region_directory / "train",
        transform=train_transform,
    )

    # Non-augmented version of train set for final evaluation.
    train_evaluation_dataset = ROIDataset(
        region_directory / "train",
        transform=evaluation_transform,
    )

    val_dataset = ROIDataset(
        region_directory / "val",
        transform=evaluation_transform,
    )

    test_dataset = ROIDataset(
        region_directory / "test",
        transform=evaluation_transform,
    )

    pin_memory = device.type == "cuda"

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    train_evaluation_loader = DataLoader(
        train_evaluation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )

    datasets = {
        "train": train_dataset,
        "train_evaluation": train_evaluation_dataset,
        "val": val_dataset,
        "test": test_dataset,
    }

    loaders = {
        "train": train_loader,
        "train_evaluation": train_evaluation_loader,
        "val": val_loader,
        "test": test_loader,
    }

    return datasets, loaders


# ============================================================
# Class weights
# ============================================================

def calculate_class_weights(
    targets: list[int],
    device: torch.device,
) -> torch.Tensor:
    """
    Balanced class weight:

        weight_c = total_samples / (num_classes × samples_in_class_c)
    """

    class_counts = np.bincount(
        targets,
        minlength=NUM_CLASSES,
    ).astype(np.float64)

    total_samples = class_counts.sum()

    weights = np.zeros(
        NUM_CLASSES,
        dtype=np.float32,
    )

    for class_id in range(NUM_CLASSES):
        if class_counts[class_id] > 0:
            weights[class_id] = (
                total_samples
                / (NUM_CLASSES * class_counts[class_id])
            )
        else:
            weights[class_id] = 0.0

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=device,
    )


# ============================================================
# Model
# ============================================================

def create_model(device: torch.device) -> nn.Module:
    model = timm.create_model(
        MODEL_NAME,
        pretrained=PRETRAINED,
        num_classes=NUM_CLASSES,
        in_chans=3,
    )

    model = model.to(device)

    return model


# ============================================================
# AMP helpers
# ============================================================

def create_grad_scaler(enabled: bool):
    """
    Support both newer and older PyTorch AMP interfaces.
    """
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(
            enabled=enabled,
        )


def autocast_context(enabled: bool):
    """
    Support both newer and older PyTorch AMP interfaces.
    """
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(
            enabled=enabled,
        )


# ============================================================
# Metric calculation
# ============================================================

def calculate_metrics(
    targets: np.ndarray,
    predictions: np.ndarray,
    probabilities: np.ndarray,
) -> dict[str, float]:
    accuracy = accuracy_score(
        targets,
        predictions,
    )

    balanced_accuracy = balanced_accuracy_score(
        targets,
        predictions,
    )

    macro_precision = precision_score(
        targets,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        targets,
        predictions,
        average="macro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        targets,
        predictions,
        average="macro",
        zero_division=0,
    )

    weighted_precision = precision_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0,
    )

    weighted_recall = recall_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        targets,
        predictions,
        average="weighted",
        zero_division=0,
    )

    try:
        macro_auc = roc_auc_score(
            targets,
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=list(range(NUM_CLASSES)),
        )
    except ValueError:
        macro_auc = float("nan")

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "macro_auc": float(macro_auc),
    }


# ============================================================
# Training and validation
# ============================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler,
    device: torch.device,
    use_amp: bool,
):
    model.train()

    total_loss = 0.0
    all_targets = []
    all_predictions = []
    all_probabilities = []

    for images, targets, _ in loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(use_amp):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = probabilities.argmax(dim=1)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        all_targets.extend(
            targets.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.detach().cpu().numpy()
        )

    average_loss = total_loss / len(loader.dataset)

    metrics = calculate_metrics(
        targets=np.asarray(all_targets),
        predictions=np.asarray(all_predictions),
        probabilities=np.asarray(all_probabilities),
    )

    metrics["loss"] = float(average_loss)

    return metrics


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool,
):
    model.eval()

    total_loss = 0.0

    all_targets = []
    all_predictions = []
    all_probabilities = []
    all_paths = []

    for images, targets, image_paths in loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            non_blocking=True,
        )

        with autocast_context(use_amp):
            logits = model(images)
            loss = criterion(logits, targets)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = probabilities.argmax(dim=1)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        all_targets.extend(
            targets.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.detach().cpu().numpy()
        )

        all_paths.extend(image_paths)

    all_targets = np.asarray(all_targets)
    all_predictions = np.asarray(all_predictions)
    all_probabilities = np.asarray(all_probabilities)

    metrics = calculate_metrics(
        targets=all_targets,
        predictions=all_predictions,
        probabilities=all_probabilities,
    )

    metrics["loss"] = float(
        total_loss / len(loader.dataset)
    )

    return {
        "metrics": metrics,
        "targets": all_targets,
        "predictions": all_predictions,
        "probabilities": all_probabilities,
        "paths": all_paths,
    }


# ============================================================
# Save plots and results
# ============================================================

def save_confusion_matrix(
    targets: np.ndarray,
    predictions: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    matrix = confusion_matrix(
        targets,
        predictions,
        labels=list(range(NUM_CLASSES)),
    )

    figure, axis = plt.subplots(
        figsize=(8, 7),
    )

    image = axis.imshow(matrix)

    figure.colorbar(
        image,
        ax=axis,
    )

    axis.set_title(
        title,
        fontsize=15,
    )

    axis.set_xlabel(
        "Predicted Class",
        fontsize=13,
    )

    axis.set_ylabel(
        "Ground-Truth Class",
        fontsize=13,
    )

    axis.set_xticks(
        np.arange(NUM_CLASSES)
    )

    axis.set_yticks(
        np.arange(NUM_CLASSES)
    )

    axis.set_xticklabels(
        CLASS_NAMES,
        rotation=25,
        ha="right",
        fontsize=11,
    )

    axis.set_yticklabels(
        CLASS_NAMES,
        fontsize=11,
    )

    threshold = (
        matrix.max() / 2.0
        if matrix.size > 0
        else 0
    )

    for row in range(NUM_CLASSES):
        for column in range(NUM_CLASSES):
            value = matrix[row, column]

            text_color = (
                "white"
                if value > threshold
                else "black"
            )

            axis.text(
                column,
                row,
                str(value),
                ha="center",
                va="center",
                fontsize=13,
                color=text_color,
            )

    figure.tight_layout()
    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


def save_normalized_confusion_matrix(
    targets: np.ndarray,
    predictions: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    matrix = confusion_matrix(
        targets,
        predictions,
        labels=list(range(NUM_CLASSES)),
        normalize="true",
    )

    matrix = np.nan_to_num(matrix)

    figure, axis = plt.subplots(
        figsize=(8, 7),
    )

    image = axis.imshow(
        matrix,
        vmin=0,
        vmax=1,
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    axis.set_title(
        title,
        fontsize=15,
    )

    axis.set_xlabel(
        "Predicted Class",
        fontsize=13,
    )

    axis.set_ylabel(
        "Ground-Truth Class",
        fontsize=13,
    )

    axis.set_xticks(
        np.arange(NUM_CLASSES)
    )

    axis.set_yticks(
        np.arange(NUM_CLASSES)
    )

    axis.set_xticklabels(
        CLASS_NAMES,
        rotation=25,
        ha="right",
        fontsize=11,
    )

    axis.set_yticklabels(
        CLASS_NAMES,
        fontsize=11,
    )

    for row in range(NUM_CLASSES):
        for column in range(NUM_CLASSES):
            value = matrix[row, column]

            text_color = (
                "white"
                if value > 0.5
                else "black"
            )

            axis.text(
                column,
                row,
                f"{value * 100:.1f}%",
                ha="center",
                va="center",
                fontsize=12,
                color=text_color,
            )

    figure.tight_layout()
    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


def save_roc_curve(
    targets: np.ndarray,
    probabilities: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    binary_targets = label_binarize(
        targets,
        classes=list(range(NUM_CLASSES)),
    )

    figure, axis = plt.subplots(
        figsize=(8, 7),
    )

    valid_curve_found = False

    for class_id, class_name in enumerate(CLASS_NAMES):
        class_targets = binary_targets[:, class_id]

        if len(np.unique(class_targets)) < 2:
            continue

        false_positive_rate, true_positive_rate, _ = roc_curve(
            class_targets,
            probabilities[:, class_id],
        )

        class_auc = roc_auc_score(
            class_targets,
            probabilities[:, class_id],
        )

        axis.plot(
            false_positive_rate,
            true_positive_rate,
            linewidth=2,
            label=f"{class_name} (AUC={class_auc:.4f})",
        )

        valid_curve_found = True

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Random classifier",
    )

    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.02)

    axis.set_xlabel(
        "False Positive Rate",
        fontsize=13,
    )

    axis.set_ylabel(
        "True Positive Rate",
        fontsize=13,
    )

    axis.set_title(
        title,
        fontsize=15,
    )

    axis.grid(alpha=0.3)

    if valid_curve_found:
        axis.legend(
            loc="lower right",
            fontsize=10,
        )

    figure.tight_layout()
    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


def save_predictions_csv(
    results: dict,
    output_path: Path,
) -> None:
    probabilities = results["probabilities"]

    columns = [
        "image_path",
        "ground_truth_id",
        "ground_truth_name",
        "predicted_id",
        "predicted_name",
        "correct",
        "probability_normal",
        "probability_stenosis",
        "probability_severe_stenosis",
    ]

    with output_path.open(
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=columns,
        )

        writer.writeheader()

        for index, image_path in enumerate(results["paths"]):
            ground_truth = int(results["targets"][index])
            prediction = int(results["predictions"][index])

            writer.writerow(
                {
                    "image_path": image_path,
                    "ground_truth_id": ground_truth,
                    "ground_truth_name": CLASS_NAMES[ground_truth],
                    "predicted_id": prediction,
                    "predicted_name": CLASS_NAMES[prediction],
                    "correct": int(ground_truth == prediction),
                    "probability_normal": float(
                        probabilities[index, 0]
                    ),
                    "probability_stenosis": float(
                        probabilities[index, 1]
                    ),
                    "probability_severe_stenosis": float(
                        probabilities[index, 2]
                    ),
                }
            )


def save_classification_report(
    targets: np.ndarray,
    predictions: np.ndarray,
    output_path: Path,
) -> None:
    report = classification_report(
        targets,
        predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=6,
        zero_division=0,
    )

    output_path.write_text(
        report,
        encoding="utf-8",
    )


def save_training_history(
    history: list[dict],
    output_path: Path,
) -> None:
    if not history:
        return

    columns = list(history[0].keys())

    with output_path.open(
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=columns,
        )

        writer.writeheader()
        writer.writerows(history)


def save_training_curves(
    history: list[dict],
    output_directory: Path,
) -> None:
    if not history:
        return

    epochs = [
        item["epoch"]
        for item in history
    ]

    train_losses = [
        item["train_loss"]
        for item in history
    ]

    val_losses = [
        item["val_loss"]
        for item in history
    ]

    figure, axis = plt.subplots(
        figsize=(8, 6),
    )

    axis.plot(
        epochs,
        train_losses,
        label="Train Loss",
        linewidth=2,
    )

    axis.plot(
        epochs,
        val_losses,
        label="Validation Loss",
        linewidth=2,
    )

    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.set_title("Training and Validation Loss")
    axis.grid(alpha=0.3)
    axis.legend()

    figure.tight_layout()
    figure.savefig(
        output_directory / "loss_curve.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)

    train_f1 = [
        item["train_macro_f1"]
        for item in history
    ]

    val_f1 = [
        item["val_macro_f1"]
        for item in history
    ]

    figure, axis = plt.subplots(
        figsize=(8, 6),
    )

    axis.plot(
        epochs,
        train_f1,
        label="Train Macro-F1",
        linewidth=2,
    )

    axis.plot(
        epochs,
        val_f1,
        label="Validation Macro-F1",
        linewidth=2,
    )

    axis.set_xlabel("Epoch")
    axis.set_ylabel("Macro-F1")
    axis.set_title("Training and Validation Macro-F1")
    axis.set_ylim(0, 1.02)
    axis.grid(alpha=0.3)
    axis.legend()

    figure.tight_layout()
    figure.savefig(
        output_directory / "macro_f1_curve.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)


# ============================================================
# Dataset information
# ============================================================

def print_dataset_information(
    region: str,
    datasets: dict,
) -> None:
    print("\n" + "=" * 78)
    print(f"DATASET: {region}")
    print("=" * 78)

    for split_name in ("train", "val", "test"):
        dataset = datasets[split_name]
        counts = Counter(dataset.targets)

        print(f"\n{split_name.upper()} SET")
        print("-" * 50)
        print(f"Total images       : {len(dataset)}")

        for class_id, class_name in enumerate(CLASS_NAMES):
            print(
                f"{class_id} - {class_name:<18}: "
                f"{counts[class_id]}"
            )


# ============================================================
# Training one anatomical region
# ============================================================

def train_region(
    region: str,
    device: torch.device,
):
    region_output_directory = OUTPUT_ROOT / region
    region_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    datasets, loaders = create_dataloaders(
        region=region,
        device=device,
    )

    print_dataset_information(
        region=region,
        datasets=datasets,
    )

    model = create_model(device)

    if USE_CLASS_WEIGHTS:
        class_weights = calculate_class_weights(
            datasets["train"].targets,
            device,
        )
    else:
        class_weights = None

    print("\nClass weights:")
    if class_weights is not None:
        for class_id, class_name in enumerate(CLASS_NAMES):
            print(
                f"  {class_name:<18}: "
                f"{class_weights[class_id].item():.6f}"
            )
    else:
        print("  Disabled")

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.05,
    )

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-6,
    )

    use_amp = (
        USE_AMP
        and device.type == "cuda"
    )

    scaler = create_grad_scaler(use_amp)

    checkpoint_path = (
        region_output_directory
        / f"best_{MODEL_NAME}_{region.lower()}.pt"
    )

    history = []

    best_score = -float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    print("\n" + "=" * 78)
    print(f"TRAINING CONVNEXT: {region}")
    print("=" * 78)

    for epoch in range(1, EPOCHS + 1):
        epoch_start_time = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=loaders["train"],
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_amp=use_amp,
        )

        validation_results = evaluate_model(
            model=model,
            loader=loaders["val"],
            criterion=criterion,
            device=device,
            use_amp=use_amp,
        )

        validation_metrics = validation_results["metrics"]

        scheduler.step()

        current_learning_rate = optimizer.param_groups[0]["lr"]
        epoch_time = time.time() - epoch_start_time

        history_row = {
            "epoch": epoch,
            "learning_rate": current_learning_rate,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics[
                "balanced_accuracy"
            ],
            "train_macro_precision": train_metrics[
                "macro_precision"
            ],
            "train_macro_recall": train_metrics["macro_recall"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_macro_auc": train_metrics["macro_auc"],
            "val_loss": validation_metrics["loss"],
            "val_accuracy": validation_metrics["accuracy"],
            "val_balanced_accuracy": validation_metrics[
                "balanced_accuracy"
            ],
            "val_macro_precision": validation_metrics[
                "macro_precision"
            ],
            "val_macro_recall": validation_metrics[
                "macro_recall"
            ],
            "val_macro_f1": validation_metrics["macro_f1"],
            "val_weighted_f1": validation_metrics["weighted_f1"],
            "val_macro_auc": validation_metrics["macro_auc"],
            "epoch_time_seconds": epoch_time,
        }

        history.append(history_row)

        current_score = validation_metrics[BEST_MODEL_METRIC]

        improved = current_score > best_score

        if improved:
            best_score = current_score
            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "epoch": epoch,
                "region": region,
                "model_name": MODEL_NAME,
                "image_size": IMAGE_SIZE,
                "num_classes": NUM_CLASSES,
                "class_names": CLASS_NAMES,
                "class_folder_to_id": CLASS_FOLDER_TO_ID,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "validation_metrics": validation_metrics,
                "class_weights": (
                    class_weights.detach().cpu()
                    if class_weights is not None
                    else None
                ),
            }

            torch.save(
                checkpoint,
                checkpoint_path,
            )

            improvement_text = " [BEST SAVED]"
        else:
            epochs_without_improvement += 1
            improvement_text = ""

        print(
            f"Epoch [{epoch:03d}/{EPOCHS:03d}] | "
            f"LR: {current_learning_rate:.7f} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Train Acc: {train_metrics['accuracy']:.4f} | "
            f"Train F1: {train_metrics['macro_f1']:.4f} | "
            f"Val Loss: {validation_metrics['loss']:.4f} | "
            f"Val Acc: {validation_metrics['accuracy']:.4f} | "
            f"Val BalAcc: "
            f"{validation_metrics['balanced_accuracy']:.4f} | "
            f"Val F1: {validation_metrics['macro_f1']:.4f} | "
            f"Time: {epoch_time:.1f}s"
            f"{improvement_text}"
        )

        save_training_history(
            history,
            region_output_directory / "training_history.csv",
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                "\nEarly stopping activated. "
                f"No validation improvement for "
                f"{EARLY_STOPPING_PATIENCE} epochs."
            )
            break

    save_training_curves(
        history=history,
        output_directory=region_output_directory,
    )

    print("\nTraining completed.")
    print(f"Best epoch          : {best_epoch}")
    print(f"Best validation F1  : {best_score:.6f}")
    print(f"Best checkpoint     : {checkpoint_path}")

    return checkpoint_path, datasets, loaders


# ============================================================
# Test saved checkpoint
# ============================================================

def evaluate_checkpoint(
    region: str,
    checkpoint_path: Path,
    datasets: dict,
    loaders: dict,
    device: torch.device,
):
    region_output_directory = OUTPUT_ROOT / region

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint was not found:\n{checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model = create_model(device)

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    class_weights = checkpoint.get("class_weights")

    if class_weights is not None:
        class_weights = class_weights.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
    )

    use_amp = (
        USE_AMP
        and device.type == "cuda"
    )

    evaluation_loaders = {
        "train": loaders["train_evaluation"],
        "val": loaders["val"],
        "test": loaders["test"],
    }

    all_split_metrics = {}

    print("\n" + "=" * 78)
    print(f"FINAL EVALUATION: {region}")
    print("=" * 78)

    for split_name, loader in evaluation_loaders.items():
        results = evaluate_model(
            model=model,
            loader=loader,
            criterion=criterion,
            device=device,
            use_amp=use_amp,
        )

        metrics = results["metrics"]
        all_split_metrics[split_name] = metrics

        split_output_directory = (
            region_output_directory
            / "evaluation"
            / split_name
        )

        split_output_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        save_predictions_csv(
            results,
            split_output_directory / "predictions.csv",
        )

        save_classification_report(
            targets=results["targets"],
            predictions=results["predictions"],
            output_path=(
                split_output_directory
                / "classification_report.txt"
            ),
        )

        save_confusion_matrix(
            targets=results["targets"],
            predictions=results["predictions"],
            output_path=(
                split_output_directory
                / "confusion_matrix.png"
            ),
            title=(
                f"{region} – {split_name.capitalize()} "
                f"Confusion Matrix"
            ),
        )

        save_normalized_confusion_matrix(
            targets=results["targets"],
            predictions=results["predictions"],
            output_path=(
                split_output_directory
                / "confusion_matrix_normalized.png"
            ),
            title=(
                f"{region} – {split_name.capitalize()} "
                f"Normalized Confusion Matrix"
            ),
        )

        save_roc_curve(
            targets=results["targets"],
            probabilities=results["probabilities"],
            output_path=(
                split_output_directory
                / "roc_curves.png"
            ),
            title=(
                f"{region} – {split_name.capitalize()} "
                f"ROC Curves"
            ),
        )

        print(f"\n{split_name.upper()} RESULTS")
        print("-" * 60)
        print(f"Images             : {len(loader.dataset)}")
        print(f"Loss               : {metrics['loss']:.6f}")
        print(f"Accuracy           : {metrics['accuracy']:.6f}")
        print(
            f"Balanced Accuracy  : "
            f"{metrics['balanced_accuracy']:.6f}"
        )
        print(
            f"Macro Precision    : "
            f"{metrics['macro_precision']:.6f}"
        )
        print(
            f"Macro Recall       : "
            f"{metrics['macro_recall']:.6f}"
        )
        print(
            f"Macro F1-score     : "
            f"{metrics['macro_f1']:.6f}"
        )
        print(
            f"Weighted Precision : "
            f"{metrics['weighted_precision']:.6f}"
        )
        print(
            f"Weighted Recall    : "
            f"{metrics['weighted_recall']:.6f}"
        )
        print(
            f"Weighted F1-score  : "
            f"{metrics['weighted_f1']:.6f}"
        )
        print(
            f"Macro AUC          : "
            f"{metrics['macro_auc']:.6f}"
        )

        print("\nPer-class results:")

        report = classification_report(
            results["targets"],
            results["predictions"],
            labels=list(range(NUM_CLASSES)),
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )

        print(report)

    metrics_path = (
        region_output_directory
        / "evaluation"
        / "all_split_metrics.json"
    )

    metrics_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with metrics_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            all_split_metrics,
            file,
            indent=4,
        )

    return all_split_metrics


# ============================================================
# Combined statistics
# ============================================================

def save_combined_test_results(
    results_by_region: dict,
) -> None:
    output_path = OUTPUT_ROOT / "combined_test_results.csv"

    columns = [
        "region",
        "loss",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "macro_auc",
    ]

    with output_path.open(
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=columns,
        )

        writer.writeheader()

        for region, split_results in results_by_region.items():
            test_metrics = split_results["test"]

            writer.writerow(
                {
                    "region": region,
                    **test_metrics,
                }
            )

    print("\n" + "=" * 100)
    print("COMBINED TEST RESULTS")
    print("=" * 100)

    header = (
        f"{'Region':<12}"
        f"{'Accuracy':>12}"
        f"{'BalAcc':>12}"
        f"{'Precision':>12}"
        f"{'Recall':>12}"
        f"{'Macro-F1':>12}"
        f"{'AUC':>12}"
    )

    print(header)
    print("-" * 100)

    for region, split_results in results_by_region.items():
        metrics = split_results["test"]

        print(
            f"{region:<12}"
            f"{metrics['accuracy']:>12.4f}"
            f"{metrics['balanced_accuracy']:>12.4f}"
            f"{metrics['macro_precision']:>12.4f}"
            f"{metrics['macro_recall']:>12.4f}"
            f"{metrics['macro_f1']:>12.4f}"
            f"{metrics['macro_auc']:>12.4f}"
        )

    print("-" * 100)
    print(f"Saved to: {output_path}")


# ============================================================
# Main
# ============================================================

def main() -> None:
    set_seed(SEED)

    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(
            f"ROI dataset directory was not found:\n"
            f"{DATASET_ROOT}"
        )

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("=" * 78)
    print("CONVNEXT ROI CLASSIFICATION")
    print("=" * 78)
    print(f"Device             : {device}")
    print(f"Dataset root       : {DATASET_ROOT}")
    print(f"Output root        : {OUTPUT_ROOT}")
    print(f"Model              : {MODEL_NAME}")
    print(f"Pretrained         : {PRETRAINED}")
    print(f"Input size         : {IMAGE_SIZE[0]} × {IMAGE_SIZE[1]}")
    print(f"Batch size         : {BATCH_SIZE}")
    print(f"Maximum epochs     : {EPOCHS}")
    print(f"Learning rate      : {LEARNING_RATE}")
    print(f"Class weighting    : {USE_CLASS_WEIGHTS}")
    print(f"AMP                : {USE_AMP and device.type == 'cuda'}")

    if device.type == "cuda":
        print(f"GPU                : {torch.cuda.get_device_name(0)}")
        print(
            f"CUDA version       : "
            f"{torch.version.cuda}"
        )

    results_by_region = {}

    for region in ANATOMICAL_REGIONS:
        region_output_directory = OUTPUT_ROOT / region

        checkpoint_path = (
            region_output_directory
            / f"best_{MODEL_NAME}_{region.lower()}.pt"
        )

        if TRAIN_MODELS:
            checkpoint_path, datasets, loaders = train_region(
                region=region,
                device=device,
            )
        else:
            datasets, loaders = create_dataloaders(
                region=region,
                device=device,
            )

            print_dataset_information(
                region=region,
                datasets=datasets,
            )

        if EVALUATE_AFTER_TRAINING:
            region_results = evaluate_checkpoint(
                region=region,
                checkpoint_path=checkpoint_path,
                datasets=datasets,
                loaders=loaders,
                device=device,
            )

            results_by_region[region] = region_results

    if results_by_region:
        save_combined_test_results(results_by_region)

    print("\n" + "=" * 78)
    print("ALL REGIONS COMPLETED")
    print("=" * 78)
    print(f"Results directory: {OUTPUT_ROOT}")


if __name__ == "__main__":
    main()

c:\Users\USER\miniconda3\envs\LSS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CONVNEXT ROI CLASSIFICATION
Device             : cuda
Dataset root       : D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\ROI_Classification_Dataset
Output root        : D:\Submitted Matrial (conference&journal)\Axial Data Artical\Code\technical validation\ConvNeXt_ROI_Classification
Model              : convnext_tiny
Pretrained         : True
Input size         : 64 × 64
Batch size         : 32
Maximum epochs     : 50
Learning rate      : 0.0001
Class weighting    : True
AMP                : True
GPU                : NVIDIA GeForce RTX 5070 Ti
CUDA version       : 12.8

DATASET: Central

TRAIN SET
--------------------------------------------------
Total images       : 1745
0 - Normal            : 1453
1 - Stenosis          : 214
2 - Severe Stenosis   : 78

VAL SET
--------------------------------------------------
Total images       : 247
0 - Normal            : 178
1 - Stenosis          : 52
2 - Severe Stenosis   : 17

TEST SET
----------------